# SIGMOD Exp 5: Symmetric Hash Join

Both sides maintain reusable state and receive updates before each round. The timed path then extracts the internal delta between consecutive readable epochs and point-probes the opposite side. `SNAP` rebuilds retained snapshots from a page-based heap MVCC base; `IVMH` keeps only the latest derived hash current and reconstructs the prior readable snapshot from the base heap when a delta comparison needs it; `EPOCH` splits once per round so delta extraction aligns with consecutive readable snapshots. This notebook wraps `symmetric_bench` and produces:

1. A stacked average per-round breakdown
2. A per-round trend plot across the configured delta rounds

The input split on the `S` side is generated once inside this notebook.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_FIXED_UPDATE_PCT,
    SIGMOD_REPEAT,
    SIGMOD_TPCH_SF,
    SIGMOD_TRIM,
    SIGMOD_WARMUP,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
    resolve_tpch_file,
    resolve_update_file,
    run_checked,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp5_symmetric_join').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

TPCH_DIR = (ROOT / 'benches' / 'sigmod' / 'tpch_data').resolve()
DELTA_S_GEN = (ROOT / 'benches' / 'exp6_symmetric_join' / 'generate_delta_s.py').resolve()
BIN = ROOT / 'target' / 'release' / 'symmetric_bench'

SF = SIGMOD_TPCH_SF
UPDATE_PCT = SIGMOD_FIXED_UPDATE_PCT
BUCKET_NUM = SIGMOD_BUCKET_NUM
WARMUP = SIGMOD_WARMUP
REPEAT = SIGMOD_REPEAT
TRIM = SIGMOD_TRIM
RUN_STAMP = current_run_stamp()
ROUNDS = 10
SPLIT_EVERY = 1
SERIES = [
    ('snap', 'nr'),
    ('ivmh', 'nr'),
    ('heap', 'wr'),
    ('chain', 'wr'),
    ('par', 'wr'),
]
STYLE = {
    ('snap', 'nr'): ('SNAP', TOL['red']),
    ('ivmh', 'nr'): ('IVMH', TOL['yellow']),
    ('heap', 'wr'): ('MONO-WR', TOL['blue']),
    ('chain', 'wr'): ('DUAL-WR', TOL['cyan']),
    ('par', 'wr'): ('EPOCH-WR', TOL['green']),
}


def normalize_result_df(df):
    df = df.copy()
    for col in ('table_type', 'repair_mode'):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
    return df


PART_FILE = resolve_tpch_file(TPCH_DIR, 'part', SF)
LINEITEM_SOURCE = resolve_tpch_file(TPCH_DIR, 'lineitem_probe', SF, '_1995-09-01_1995-10-01.tbl')
DELTA_R_FILE = resolve_update_file(TPCH_DIR, SF, UPDATE_PCT, 'uniform')
LINEITEM_INITIAL = DATA_DIR / 'lineitem_initial.tbl'
LINEITEM_DELTA = DATA_DIR / 'lineitem_delta.tbl'

print('PART       :', PART_FILE)
print('LINEITEM    :', LINEITEM_SOURCE)
print('DELTA R     :', DELTA_R_FILE)
print('BIN         :', BIN)
print('STAMP       :', RUN_STAMP)

In [ ]:
print('Building symmetric_bench...')
run_checked(['cargo', 'build', '--release', '--bin', 'symmetric_bench'], ROOT)
print('Build OK')

In [ ]:
if not LINEITEM_INITIAL.exists() or not LINEITEM_DELTA.exists():
    print('Generating delta-S split...')
    run_checked([sys.executable, str(DELTA_S_GEN), str(LINEITEM_SOURCE), str(DATA_DIR), '--initial_pct', '80', '--seed', '42'], ROOT)
else:
    print('Reusing existing delta-S split')

for path in [LINEITEM_INITIAL, LINEITEM_DELTA]:
    print(path.name, 'rows =', sum(1 for _ in open(path)))

In [ ]:
def run_symmetric(table_type, repair_mode, output_csv, per_round_csv):
    result = run_checked([
        str(BIN),
        '--r-file', str(PART_FILE),
        '--s-file', str(LINEITEM_INITIAL),
        '--delta-r-file', str(DELTA_R_FILE),
        '--delta-s-file', str(LINEITEM_DELTA),
        '--table-type', table_type,
        '--repair-mode', repair_mode,
        '--bucket-num', str(BUCKET_NUM),
        '--rounds', str(ROUNDS),
        '--warmup', str(WARMUP),
        '--repeat', str(REPEAT),
        '--trim', str(TRIM),
        '--split-every', str(SPLIT_EVERY),
        '--output-csv', str(output_csv),
        '--per-round-csv', str(per_round_csv),
    ], ROOT, quiet=True)
    for line in result.stderr.splitlines():
        if 'total_ms' in line or 'build_' in line or 'round ' in line:
            print(' ', line)

AVG_CSV = DATA_DIR / f'sigmod_exp5_avg_{RUN_STAMP}.csv'
PER_ROUND_CSV = DATA_DIR / f'sigmod_exp5_per_round_{RUN_STAMP}.csv'
for path in [AVG_CSV, PER_ROUND_CSV]:
    if path.exists():
        path.unlink()

for table_type, repair_mode in SERIES:
    print(f'run: {table_type}/{repair_mode}')
    run_symmetric(table_type, repair_mode, AVG_CSV, PER_ROUND_CSV)

df_avg = normalize_result_df(pd.read_csv(AVG_CSV))
df_round = normalize_result_df(pd.read_csv(PER_ROUND_CSV))
display(df_avg)

In [ ]:
PHASES = [
    ('avg_delta_r_extract_ms', 'ΔR Extract', TOL['yellow'], True),
    ('avg_delta_r_probe_ms', 'ΔR Probe', TOL['red'], False),
    ('avg_delta_s_extract_ms', 'ΔS Extract', TOL['blue'], True),
    ('avg_delta_s_probe_ms', 'ΔS Probe', TOL['green'], False),
]
HATCH = {'ΔR Probe': '///', 'ΔS Probe': '\\\\'}

fig, ax = plt.subplots(1, 1, figsize=(7.2, 4.4))

x = []
labels = []
for idx, key in enumerate(SERIES):
    table_type, repair_mode = key
    label, color = STYLE[key]
    sub = df_avg[(df_avg['table_type'] == table_type) & (df_avg['repair_mode'] == repair_mode)]
    if sub.empty:
        raise ValueError(f'Missing average row for {table_type}/{repair_mode}')
    row = sub.iloc[0]
    x.append(idx)
    labels.append(label)
    bottom = 0.0
    for col, phase_label, phase_color, is_write in PHASES:
        value = float(row[col])
        if is_write:
            ax.bar(idx, value, bottom=bottom, width=0.62, color=phase_color, edgecolor='black', linewidth=0.4)
        else:
            ax.bar(idx, value, bottom=bottom, width=0.62, color='white', edgecolor='black', linewidth=0.4)
            ax.bar(idx, value, bottom=bottom, width=0.62, color='none', edgecolor=phase_color, linewidth=0.9, hatch=HATCH[phase_label])
        bottom += value

ax.set_xticks(x, labels)
ax.set_ylabel('Duration (ms)')
ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.6)
legend_handles = []
legend_labels = []
for col, phase_label, phase_color, is_write in PHASES[::-1]:
    if is_write:
        patch = Patch(facecolor=phase_color, edgecolor='black', linewidth=0.4)
    else:
        patch = Patch(facecolor='white', edgecolor=phase_color, linewidth=0.9, hatch=HATCH[phase_label])
    legend_handles.append(patch)
    legend_labels.append(phase_label)
ax.legend(legend_handles, legend_labels, title='Operations', loc='upper right', ncol=2, framealpha=0.95)

fig.tight_layout()
out_pdf = FIGS_DIR / f'sigmod_exp5_symmetric_join_{RUN_STAMP}.pdf'
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)
